In [43]:
import pandas as pd
import numpy as np
import pulp
import os
import joblib

## 1. Load Data & Trained Models
Load `dataset_transformed.csv` (hasil dari pipeline feature engineering) dan model XGBoost yang sudah di-train di notebook `model.ipynb`. Juga load metadata `diff_order` per komoditas untuk inverse differencing.

In [44]:
MODEL_DIR = 'trained_models'
predicted_prices_tomorrow = {}

# Load dataset yang sama dengan yang digunakan saat training
df = pd.read_csv('dataset_transformed.csv')

# Load metadata diff_order per komoditas
metadata_komoditas = joblib.load(os.path.join(MODEL_DIR, 'metadata_komoditas.pkl'))

# Identifikasi fitur (sama persis dengan notebook model.ipynb)
exclude_cols = ['Unnamed: 0', 'variant_id', 'variant_nama', 'satuan_display', 'tanggal', 'harga', 'harga_final', 'diff_order']
features = [col for col in df.columns if col not in exclude_cols]

print(f"Features yang digunakan: {features}")
print(f"Metadata diff_order: {metadata_komoditas}")

Features yang digunakan: ['day', 'day_of_week', 'week_of_year', 'is_month_start', 'is_weekend', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_8', 'lag_9', 'lag_11', 'lag_14', 'rolling_mean_3d', 'rolling_mean_7d']
Metadata diff_order: {'Beras SPHP Bulog': 0, 'Cabai Merah Keriting': 0, 'Cabai Merah Besar': 1, 'Cabai Rawit Merah': 1, 'Bawang Merah': 1, 'Gula Pasir Curah': 1, 'Minyak Goreng Sawit Curah': 1, 'Minyak Goreng Sawit Kemasan Premium': 1, 'Minyakita': 1, 'Daging Sapi Paha Belakang': 1, 'Telur Ayam Ras': 2, 'Tepung Terigu': 1, 'Daging Ayam Ras': 1, 'Bawang Putih Honan': 1, 'Kedelai Impor': 1, 'Beras Premium': 1, 'Beras Medium': 1}


In [45]:
# Loop semua komoditas, muat modelnya, dan prediksi harga besok
all_commodities = df['variant_nama'].unique()

for commodity in all_commodities:
    safe_filename = commodity.replace(' ', '_').replace('/', '_')
    model_path = os.path.join(MODEL_DIR, f'xgb_model_{safe_filename}.pkl')
    
    if os.path.exists(model_path):
        model = joblib.load(model_path)
        
        # Ambil data komoditas ini, urut berdasarkan tanggal
        df_var = df[df['variant_nama'] == commodity].sort_values('tanggal').copy()
        
        # Ambil diff_order dari metadata
        diff_val = metadata_komoditas.get(commodity, 0)
        
        # Pre-processing features (sama seperti saat training)
        df_var['harga_shift_1'] = df_var['harga'].shift(1)
        df_var['harga_shift_2'] = df_var['harga'].shift(2)
        df_var['sma_7_hari'] = df_var['harga_shift_1'].rolling(window=7).mean()
        
        # Ambil baris data terakhir untuk prediksi
        latest_features = df_var[features].iloc[-1:].copy()
        
        # Prediksi harga_final (mungkin dalam bentuk diff)
        y_pred_diff = model.predict(latest_features)[0]
        
        # Inverse Differencing untuk mendapatkan harga rupiah
        last_row = df_var.iloc[-1]
        harga_shift_1 = last_row['harga']
        harga_shift_2 = df_var['harga'].iloc[-2] if len(df_var) >= 2 else harga_shift_1
        
        if diff_val == 0:
            # Tidak ada differencing, prediksi langsung = harga
            tomorrow_price = y_pred_diff
        elif diff_val == 1:
            # 1st order diff: harga = harga_kemarin + prediksi_diff
            tomorrow_price = harga_shift_1 + y_pred_diff
        elif diff_val == 2:
            # 2nd order diff: harga = harga_kemarin + (harga_kemarin - harga_2_hari_lalu) + prediksi_diff
            tomorrow_price = harga_shift_1 + (harga_shift_1 - harga_shift_2) + y_pred_diff
        else:
            tomorrow_price = y_pred_diff
        
        # Pastikan harga tidak negatif
        tomorrow_price = max(0, float(tomorrow_price))
        
        predicted_prices_tomorrow[commodity] = tomorrow_price
        
print(f"\nSUCCESS: Berhasil memprediksi harga untuk {len(predicted_prices_tomorrow)} komoditas.")
print("\nPrediksi Harga Besok (Rp/kg):")
for name, price in predicted_prices_tomorrow.items():
    print(f"  - {name:<40s}: Rp {price:>10,.2f}")


SUCCESS: Berhasil memprediksi harga untuk 17 komoditas.

Prediksi Harga Besok (Rp/kg):
  - Beras SPHP Bulog                        : Rp  12,367.76
  - Cabai Merah Keriting                    : Rp  35,958.39
  - Cabai Merah Besar                       : Rp  38,755.04
  - Cabai Rawit Merah                       : Rp  60,381.63
  - Bawang Merah                            : Rp  40,557.09
  - Gula Pasir Curah                        : Rp  18,301.78
  - Minyak Goreng Sawit Curah               : Rp  19,440.31
  - Minyak Goreng Sawit Kemasan Premium     : Rp  21,716.00
  - Minyakita                               : Rp  15,993.98
  - Daging Sapi Paha Belakang               : Rp 138,118.42
  - Telur Ayam Ras                          : Rp  29,572.27
  - Tepung Terigu                           : Rp  12,455.75
  - Daging Ayam Ras                         : Rp  38,089.29
  - Bawang Putih Honan                      : Rp  35,776.21
  - Kedelai Impor                           : Rp  13,442.98
  - Beras Pr

## 2. Load Data Nutrisi & Persiapan Optimasi
Load data gizi per komoditas dan siapkan item yang tersedia untuk optimasi diet.

In [46]:
df_nutrition = pd.read_csv('dataset_gizi.csv')
df_nutrition.set_index('KOMODITAS', inplace=True)

available_items = [item for item in predicted_prices_tomorrow.keys() if item in df_nutrition.index]

print(f"Komoditas tersedia untuk optimasi: {len(available_items)}")
for item in available_items:
    print(f"  - {item}")

Komoditas tersedia untuk optimasi: 16
  - Cabai Merah Keriting
  - Cabai Merah Besar
  - Cabai Rawit Merah
  - Bawang Merah
  - Gula Pasir Curah
  - Minyak Goreng Sawit Curah
  - Minyak Goreng Sawit Kemasan Premium
  - Minyakita
  - Daging Sapi Paha Belakang
  - Telur Ayam Ras
  - Tepung Terigu
  - Daging Ayam Ras
  - Bawang Putih Honan
  - Kedelai Impor
  - Beras Premium
  - Beras Medium


## 3. Definisi Variabel Keputusan & Kendala
Menggunakan PuLP untuk Linear Programming: meminimalkan biaya pangan harian per anak dengan memenuhi kebutuhan gizi minimum.

In [47]:
# Membuat model untuk meminimalkan biaya (LpMinimize)
lp_problem = pulp.LpProblem("NutriCost_Diet_Optimization", pulp.LpMinimize)

# Mendefinisikan Variabel Keputusan "Berapa Kg bahan X yang harus dibeli?"
# lowBound=0 memastikan tidak membeli bahan dalam jumlah minus
x = pulp.LpVariable.dicts("qty_kg", available_items, lowBound=0, cat='Continuous')

pakai_minyak_curah   = pulp.LpVariable('Pakai_Minyak_Curah',   cat='Binary')
pakai_minyak_premium = pulp.LpVariable('Pakai_Minyak_Premium', cat='Binary')
pakai_minyakita      = pulp.LpVariable('Pakai_Minyakita',      cat='Binary')

pakai_beras_premium  = pulp.LpVariable('Pakai_Beras_Premium',  cat='Binary')
pakai_beras_medium   = pulp.LpVariable('Pakai_Beras_Medium',   cat='Binary')

In [48]:
# 1. FUNGSI OBJEKTIF: Total Biaya = Harga * Kuantitas (x diasumsikan dalam Kg)
lp_problem += pulp.lpSum([predicted_prices_tomorrow[i] * x[i] for i in available_items]), "Total_Cost"

# 2. KENDALA GIZI 1: Total Energi Minimal 700 Kkal (Porsi Anak 1x Makan)
lp_problem += pulp.lpSum([df_nutrition.loc[i, 'Energi'] * x[i] for i in available_items]) >= 700, "Min_Energy"

# 3. KENDALA GIZI 2: Total Protein Minimal 20 gram 
lp_problem += pulp.lpSum([df_nutrition.loc[i, 'Protein'] * x[i] for i in available_items]) >= 20, "Min_Protein"

# 4. KENDALA LOGIKA MANUSIA (Batas maksimal & minimal absolut dalam Kg)
for i in available_items:
    if 'Beras' in i:
        lp_problem += x[i] <= 0.080  # Maksimal 80g
        # NOTE: Lower bound ditangani secara kondisional di bawah via Big-M
        
    elif 'Terigu' in i:
        lp_problem += x[i] <= 0.015  # Maksimal 15g
        
    elif 'Kedelai' in i:
        lp_problem += x[i] >= 0.020  # Wajib ada nabati minimal 20g
        lp_problem += x[i] <= 0.040  # Maksimal 40g
        
    elif 'Ayam' in i or 'Telur' in i:
        lp_problem += x[i] <= 0.060  # Maksimal 60g
        
    elif 'Sapi' in i:
        lp_problem += x[i] <= 0.050  # Maksimal 50g
        
    elif 'Gula' in i:
        lp_problem += x[i] <= 0.008  # Maksimal 8g
        
    elif 'Minyak' in i:
        lp_problem += x[i] <= 0.010  # Maksimal 10g
        # NOTE: Lower bound ditangani secara kondisional di bawah via Big-M
        
    elif 'Bawang' in i:
        lp_problem += x[i] >= 0.002  # Minimal bumbu 2g
        lp_problem += x[i] <= 0.006  # Maksimal 6g
        
    elif 'Cabai' in i:
        lp_problem += x[i] <= 0.002  # Maksimal 2g (Toleransi pedas anak)

# Memaksa harus ada Protein Hewani (Ayam / Sapi / Telur) minimal 30 gram (0.030 kg)
protein_hewani_items = [i for i in available_items if 'Ayam' in i or 'Sapi' in i or 'Telur' in i]
if protein_hewani_items:
    lp_problem += pulp.lpSum([x[i] for i in protein_hewani_items]) >= 0.030, "Min_Protein_Hewani"

# ── CONSTRAINT MUTUAL EXCLUSION: Pilih Maks 1 Jenis per Grup ─────────────
lp_problem += pakai_minyak_curah + pakai_minyak_premium + pakai_minyakita <= 1, 'Pilih_Maks_1_Minyak'
lp_problem += pakai_beras_premium + pakai_beras_medium <= 1,                    'Pilih_Maks_1_Beras'

# ── CONSTRAINT BIG-M: Hubungkan biner ke variabel gramatur ───────────────
# Optimalisasi: Gunakan batas atas (Upper Bound) spesifik sebagai nilai M, bukan M=1.0.
# Ini membuat ruang komputasi PuLP jauh lebih sempit dan cepat.

if 'Minyak Goreng Sawit Curah' in available_items:
    lp_problem += x['Minyak Goreng Sawit Curah']           <= 0.010 * pakai_minyak_curah,   'BigM_Minyak_Curah'
if 'Minyak Goreng Sawit Kemasan Premium' in available_items:
    lp_problem += x['Minyak Goreng Sawit Kemasan Premium'] <= 0.010 * pakai_minyak_premium, 'BigM_Minyak_Premium'
if 'Minyakita' in available_items:
    lp_problem += x['Minyakita']                           <= 0.010 * pakai_minyakita,      'BigM_Minyakita'

if 'Beras Premium' in available_items:
    lp_problem += x['Beras Premium'] <= 0.080 * pakai_beras_premium, 'BigM_Beras_Premium'
if 'Beras Medium' in available_items:
    lp_problem += x['Beras Medium']  <= 0.080 * pakai_beras_medium,  'BigM_Beras_Medium'

# ── FIX: CONDITIONAL LOWER BOUND (via Big-M) ─────────────────
# Jika biner=1 -> x >= Batas Minimal; jika biner=0 -> x >= 0

# Kondisi Beras (Minimal 50g)
if 'Beras Premium' in available_items:
    lp_problem += x['Beras Premium'] >= 0.050 * pakai_beras_premium, 'MinBeras_Premium'
if 'Beras Medium' in available_items:
    lp_problem += x['Beras Medium']  >= 0.050 * pakai_beras_medium,  'MinBeras_Medium'

# Kondisi Minyak (Minimal 2g agar tidak masak tanpa minyak jika dipilih)
if 'Minyak Goreng Sawit Curah' in available_items:
    lp_problem += x['Minyak Goreng Sawit Curah']           >= 0.002 * pakai_minyak_curah,   'MinMinyak_Curah'
if 'Minyak Goreng Sawit Kemasan Premium' in available_items:
    lp_problem += x['Minyak Goreng Sawit Kemasan Premium'] >= 0.002 * pakai_minyak_premium, 'MinMinyak_Premium'
if 'Minyakita' in available_items:
    lp_problem += x['Minyakita']                           >= 0.002 * pakai_minyakita,      'MinMinyakita'

## 4. Solve & Tampilkan Hasil Optimasi

In [49]:
lp_problem.solve()

# Cek apakah mesin berhasil menemukan jalan keluar
if pulp.LpStatus[lp_problem.status] == 'Optimal':
    total_cost = pulp.value(lp_problem.objective)
    
    print(f"Total Biaya per Anak : Rp {total_cost:,.2f}")
    print("\nRekomendasi Resep & Gramatur")
    
    total_kcal = 0
    total_protein = 0
    
    for item in available_items:
        qty_kg = x[item].varValue
        if qty_kg > 0.001:  # Hanya tampilkan bahan yang dipakai lebih dari 1 gram
            qty_grams = qty_kg * 1000
            kcal_supplied = qty_kg * df_nutrition.loc[item, 'Energi']
            protein_supplied = qty_kg * df_nutrition.loc[item, 'Protein']
            cost_item = qty_kg * predicted_prices_tomorrow[item]
            
            print(f"- {item:.<30} {qty_grams:>5.0f} gram  (Rp {cost_item:>6,.0f})")
            
            total_kcal += kcal_supplied
            total_protein += protein_supplied
            
    print("\n[ Pencapaian Target Gizi ]")
    print(f"- Total Energi Terpenuhi   : {total_kcal:.1f} Kcal")
    print(f"- Total Protein Terpenuhi  : {total_protein:.1f} gram")
else:
    print(f"ERROR: Tidak dapat menemukan menu optimal (Status: {pulp.LpStatus[lp_problem.status]})")

Total Biaya per Anak : Rp 3,903.93

Rekomendasi Resep & Gramatur
- Bawang Merah..................     2 gram  (Rp     81)
- Gula Pasir Curah..............     8 gram  (Rp    146)
- Minyakita.....................    10 gram  (Rp    160)
- Tepung Terigu.................    15 gram  (Rp    187)
- Daging Ayam Ras...............    43 gram  (Rp  1,623)
- Bawang Putih Honan............     2 gram  (Rp     72)
- Kedelai Impor.................    40 gram  (Rp    538)
- Beras Medium..................    80 gram  (Rp  1,097)

[ Pencapaian Target Gizi ]
- Total Energi Terpenuhi   : 700.0 Kcal
- Total Protein Terpenuhi  : 28.0 gram
